# Data Preparation
This notebook mirrors `pipeline.data_preparation`. It follows the CRISP-DM
data-preparation tasks: select data, clean data, construct data, integrate
data, and format data.



In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pipeline").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

from pipeline.data_preparation import prepare_modeling_data

DATASET_VERSION = "dataset_modeling_v0.2"
OUTPUT_DIR = PROJECT_ROOT / "data/processed/modeling_v0.2"
RUN_PREPARATION = False

if RUN_PREPARATION:
    prepare_modeling_data(
        raw_path=PROJECT_ROOT / "enriched_prs_raw_new.jsonl",
        output_dir=OUTPUT_DIR,
        config_path=PROJECT_ROOT / "config.yaml",
        token_cap=1000,
        batch_size=32,
        dataset_version=DATASET_VERSION,
    )



## Select Data
The raw enriched pull request records are split by repository. The modeling
target is `review_concern`; rows without review activity are excluded from
train/validation/test matrices.



In [2]:
summary = json.loads(
    (OUTPUT_DIR / f"{DATASET_VERSION}.preparation_summary.json").read_text()
)
manifest = json.loads(
    (OUTPUT_DIR / f"{DATASET_VERSION}.feature_manifest.json").read_text()
)
summary["split_summary"]



{'all': {'rows': 10000,
  'repos': 6033,
  'review_concern_0': 446,
  'review_concern_1': 9545,
  'accepted_quality_rows': 7373},
 'train': {'rows': 8056,
  'repos': 4820,
  'review_concern_0': 367,
  'review_concern_1': 7689,
  'accepted_quality_rows': 5924},
 'val': {'rows': 954,
  'repos': 602,
  'review_concern_0': 39,
  'review_concern_1': 915,
  'accepted_quality_rows': 712},
 'test': {'rows': 981,
  'repos': 604,
  'review_concern_0': 40,
  'review_concern_1': 941,
  'accepted_quality_rows': 737}}

## Clean Data
Bot authors, documentation-only changes, generated/vendor-only changes,
and missing source patches are retained as quality metadata. These fields are
kept for audit and are not used as model features.



In [3]:
pd.DataFrame(summary["top_reject_reasons"], columns=["reason", "rows"])



,reason,rows
0,no_source_patches,1641
1,no_source_files,1639
2,not_enough_discussion,578
3,not_enough_meaningful_review_comments,535
4,docs_only,288
5,bot_author,70
6,not_enough_review_comments,43
7,generated_or_vendor_only,28
8,bad_diff_size,9
9,bad_changed_files_count,7


## Construct Data
The table contains pull-request-intrinsic tabular features plus a 768-dim
ModernBERT embedding of source-code `files[].patch`, capped to 1,000 tokens
per row.



In [4]:
print("target:", manifest["target_column"])
print("feature_count:", len(manifest["feature_columns"]))
print("embedding_columns:", len(manifest["embedding_columns"]))
print("token_cap:", manifest["embedding_token_cap"])



target: review_concern
feature_count: 802
embedding_columns: 768
token_cap: 1000


## Integrate Data
Pull request metadata, changed-file summaries, source patches, labels, and
quality metadata are integrated into one row per pull request.



In [5]:
all_df = pd.read_parquet(
    OUTPUT_DIR / f"{DATASET_VERSION}.all.parquet",
    columns=[
        "example_id",
        "repo",
        "pr_number",
        "split",
        "review_concern",
        "accepted_quality",
        "top_language",
        "source_patch_token_count_capped",
    ],
)
all_df.head()



,example_id,repo,pr_number,split,review_concern,accepted_quality,top_language,source_patch_token_count_capped
0,equinor__acidwatch__pull_336,equinor/acidwatch,336,train,1.0,1,python,1000
1,CliMA__ClimaOcean.jl__pull_475,CliMA/ClimaOcean.jl,475,train,1.0,0,unknown,0
2,microsoft__PowerToys__pull_38027,microsoft/PowerToys,38027,train,1.0,1,c#,1000
3,envoyproxy__gateway__pull_5569,envoyproxy/gateway,5569,test,1.0,1,go,1000
4,pagopa__io-app__pull_6977,pagopa/io-app,6977,test,1.0,1,typescript,1000


## Format Data
The `.npz` files contain numeric `X` and `y` arrays and can be passed directly
to scikit-learn style `.fit()` and `.predict()` calls.



In [6]:
train = np.load(OUTPUT_DIR / f"{DATASET_VERSION}.train.npz")
val = np.load(OUTPUT_DIR / f"{DATASET_VERSION}.val.npz")
print("train:", train["X"].shape, train["y"].shape, np.bincount(train["y"]))
print("val:", val["X"].shape, val["y"].shape, np.bincount(val["y"]))

model = DummyClassifier(strategy="most_frequent")
model.fit(train["X"], train["y"])
pred = model.predict(val["X"])
print("fit_predict_ok:", pred.shape)


train: (8056, 802) (8056,) [ 367 7689]
val: (954, 802) (954,) [ 39 915]


fit_predict_ok: (954,)
